# Lesson 05: When to commit

> **Question.** When should we buy the site, start Building B, or stop?
>
> **You will learn to:**
>
> 1. write a decision rule with a trigger on an observed value;
> 2. use only the information available at the decision date;
> 3. check funding before a commitment, and read a decision trace;
> 4. compare the fixed plan with a strategy on the same paths, as paired differences;
> 5. tell rule parameters from realised decisions.
>
> **Before you start.** Lessons 03 and 04.
>
> **Time.** 60 minutes core, 30 minutes optional (several triggers).
>
> **Not in this lesson.** Changes to the design (lesson 07).
>
> **Source.** Figure 9.1 in de Neufville and Geltner (2018), used with the
> publisher's permission and with attribution. Chapters 9 and 10 are cited for the
> method.

## 1. The plan, the paths and the dates that can move

Start from lesson 04's Model and paths. The plan's dates are related:
the shared works start on the start date, Building A follows a year later, and
Building B cannot start before the shared works finish. These equations are part of
the Model. Only two dates are inputs: the start date and Building B's start.

In [ ]:
from datetime import date

import helpers

import rangekeeper as rk

In [ ]:
model = rk.io.load(path=helpers.data_path(name="04_base_plan_on_paths.model.json"))
paths = helpers.load_paths(name="04_rent_index.paths.json")
model.equations.where(formulation="Dates")  # the plan's date equations

In [ ]:
fixed = rk.io.load(path=helpers.data_path(name="04_base_case.specification.json"))
fixed.roles  # lesson 04's base case: start in 2027, Building B in 2030

Lesson 04 ended with a funding shortfall on every path: the plan needs more
funding, whichever path occurs. So this lesson raises the loan limit to 90e6 AUD, for
the fixed plan and for each strategy. The change is in this lesson only: lessons 03 and
04 and the data package keep their limit.

In [ ]:
fixed = fixed.fix(loan_limit="90e6 AUD")

> **Finance note.** This Model has no fee and no pricing for a larger loan. Interest
> stays at the same rate, and the limit is not an amount that the plan draws. So the
> higher limit does not change the NPV of a plan. It changes only the funding that a
> rule can read. A real lender would charge for it.

## 2. Three readable histories

Before 500 paths, look at three: an early recovery, a long downturn,
and a short recovery then a decline. The fixed plan starts in 2027 on all three.

In [ ]:
stories = rk.io.load(path=helpers.data_path(name="05_histories.paths.json"))
helpers.plot_paths(paths=stories, names=["rent_index"])

In [ ]:
fixed_stories_outcome = rk.execution.solve(specification=fixed.across(paths=stories))
fixed_stories_outcome.table("equity_npv", "peak_shortfall")

## 3. A rule: wait to start

In place of a fixed start date, fix it with a rule. From 2027 to 2030,
read the latest rent index as soon as it is known, each January. Start on the first
date when it reaches 1.05. If it never does, do not buy the site. A rule is a query
over a window of dates: the first date in the window where a condition holds, or a
default. A window acts each time a value that the condition reads becomes known;
`earliest` and `latest` bound it. The trigger, 1.05, is a value in the Specification,
so you can change it later without rewriting the rule. `rk.duration.NEVER` means the
event does not happen. The paths start in 2027, so the 2027 index is the first value
known: the first check is on 2028-01-01, and no value is known on 2027-01-01.

In [ ]:
start_window = rk.specification.Window(
    earliest=date(2027, 1, 1), latest=date(2030, 1, 1)
)
start_rule = start_window.where(condition="rent_index >= start_trigger").first(
    otherwise=rk.duration.NEVER
)
wait = fixed.parameters(start_trigger=1.05).fix(start_date=start_rule)
wait_outcome = rk.execution.solve(specification=wait.across(paths=stories))
wait_outcome.table("start_date", "b_start", "equity_npv")

The trace shows each date in the window: what was observed, when it
became known, and the result. Building B's start is still fixed at 2030. **Predict:** what
happens on a history where the project starts in 2030? Building B would start
before the shared works finish. The Model's constraint `b_start >= shared_start +
1 year` catches this: the case is infeasible, and the diagnostic names the
constraint. The rule for Building B must follow the rule for the start.

In [ ]:
downturn = wait_outcome.case(key="long downturn")
downturn.trace  # each window date: observed value, when it was known, result

In [ ]:
downturn.status, downturn.diagnostics

> **Finance note.** A start date that follows the market is an option to wait. Its
> value comes from avoiding bad outcomes. Its cost is lost income and a shorter
> operating period before the common sale date.

## 4. Only what is known

A rule cannot use the future. The 2028 rent index is known only at the
end of 2028. A rule that reads next year's index fails when the Specification is
made, before any solve. No date is known at that time, so the error names the rule and
prints `None` for `available_on` and `decision_date`.

In [ ]:
try:
    look_ahead = start_window.where(condition="rent_index.shift(by=-1) >= 1.05").first(
        otherwise=rk.duration.NEVER
    )
    fixed.fix(start_date=look_ahead)
except rk.errors.AvailabilityError as error:
    print(error.code, error.key, error.available_on, error.decision_date)

> **Finance note.** Hindsight bias: a rule that sees the future always looks
> valuable. Availability dates come from the market paths (lesson 04).

## 5. Start Building B when the market and funding allow

A rule for Building B's start too. Start B when the index reaches its
own trigger, 1.10, and the funding allows it. The funding check counts what is still
due. When B starts, the project owes B's whole budget and what it has not yet paid on
the shared works and on Building A. Add that amount to the Model as `still_due`.

In [ ]:
draft = model.edit()
project = draft.entity(code="project")
funding = project.formulation(name="Funding")
funding.equations(
    "still_due = (shared_works.cumulative() + a_works.cumulative() - shared_works.total() - a_works.total()).since(on=start_date)",
)
model = draft.commit()

The available equity and the available loan must cover `b_cost + still_due`.
All of these come from lesson 03's funding equations, with the decisions made so far.
Building B cannot start before the shared works finish, so the window's `earliest`
is an expression: on each path, it skips the dates before the shared works finish.

In [ ]:
b_window = rk.specification.Window(
    earliest="shared_start + 1 year", latest=date(2032, 1, 1)
)
b_rule = b_window.where(
    condition="rent_index >= b_start_trigger and equity_available + loan_available >= b_cost + still_due"
).first(otherwise=rk.duration.NEVER)
start_b = wait.on(model=model).parameters(b_start_trigger=1.10).fix(b_start=b_rule)
# the decided values, their rules and their windows
helpers.show_decisions(of=start_b)

Read what this check promises. On the decision date, the funding that is
known covers the budgets that are still due. It promises nothing about later dates. It
does not count loan interest, operating costs, renewals or the income to come. After
the decision, a shock can lower the income, and interest and costs do not fall with it.
The loan can then go above its limit. `commitment_gap` shows this too: it is the largest
amount of `committed - equity_available - loan_available` on any date, and when the works
are paid, that is the amount of the loan above its limit. Section 7 reads how often this
happens.

## 6. Stop and sell Building B's site

If the project started and Building B has not started by 2033, sell B's
share of the site. A project that never started never bought the site, so it has
nothing to sell. The price rule is part of the modelling: the site sells at its share of the land cost,
scaled by the rent index on the sale date. Add it to the Model. The sale date is a
decided value. The sale is an event, so a site that is not sold has no sale: with
`NEVER`, the event is zero in every period.

In [ ]:
yearly = rk.duration.Frequency.YEAR
years = rk.duration.Span(start=date(2027, 1, 1), end=date(2038, 1, 1)).periods(
    frequency=yearly
)
draft = model.edit()
project = draft.entity(code="project")
project.value(key="b_land_share", units="ratio")
project.date(key="b_sale_date")
project.flow(key="b_site_sale", units="AUD", periods=years)
project.equations(
    "b_site_sale = rx.flow.event(amount=b_land_share * land_cost, on=b_sale_date) * rent_index",
)
project.add_to(key="project_cash", term="b_site_sale")
model = draft.commit()

In [ ]:
sale_window = rk.specification.Window(
    earliest=date(2033, 1, 1), latest=date(2033, 1, 1)
)
sale_rule = sale_window.where(
    condition="start_date != NEVER and b_start == NEVER"
).first(otherwise=rk.duration.NEVER)
strategy = start_b.on(model=model).fix(b_land_share="40 %", b_sale_date=sale_rule)

In [ ]:
stories_outcome = rk.execution.solve(specification=strategy.across(paths=stories))
stories_outcome.decisions  # one row for each case and window date

In [ ]:
# each history and its decisions
helpers.plot_decisions(outcome=stories_outcome, key="rent_index")

## 7. All paths: paired comparison

**Predict:** does the strategy raise the mean equity NPV? Run the fixed
plan and the strategy on the same paths, both with the higher loan limit.
Compare them path by path: each paired difference holds the market fixed and shows the
effect of the rules alone.

In [ ]:
fixed_plan = fixed.on(model=model).fix(
    b_land_share="40 %", b_sale_date=rk.duration.NEVER
)
fixed_outcome = rk.execution.solve(specification=fixed_plan.across(paths=paths))
strategy_outcome = rk.execution.solve(specification=strategy.across(paths=paths))

In [ ]:
difference = rk.execution.compare(
    outcomes={"Strategy": strategy_outcome, "Fixed plan": fixed_outcome},
    keys=["equity_npv", "peak_shortfall"],
    baseline="Fixed plan",
)
difference.summary()  # mean difference, share of paths better, worst loss

Then read each plan alone: the mean, the spread, the worst path and the
best path.

In [ ]:
fixed_equity = fixed_outcome.distribution(key="equity_npv")
fixed_equity.summary()

In [ ]:
strategy_equity = strategy_outcome.distribution(key="equity_npv")
strategy_equity.summary()

In [ ]:
strategy_outcome.distribution(key="b_start").counts()

In [ ]:
(
    strategy_equity.mean() > fixed_equity.mean(),
    strategy_equity.quantile(level=0) > fixed_equity.quantile(level=0),
    strategy_equity.quantile(level=1) > fixed_equity.quantile(level=1),
    fixed_equity.probability(above="0 AUD"),
    strategy_equity.probability(above="0 AUD"),
)

**The answer.** The last cell answers the prediction on the paths that you ran:
its first value is true only if the strategy raises the mean, the next two compare the
worst and the best path, and the last two are the shares of paths above zero. On the
lesson's 500 paths, the answer is no. At these triggers, the strategy has a lower mean
equity NPV than the fixed plan. It also has a smaller share of paths above zero. These
rules do not show that flexibility pays. A quick run with `RK_PATH_COUNT` has few paths,
so its values can differ.

The strategy gains only at the two ends: on the worst path and on the best path. Its 5 %
quantile is almost the same as the fixed plan's, and its 95 % quantile is lower. Read the
worst path with care. On the 500 paths, the worst path of each plan loses all of the
equity, 50e6 AUD. The strategy's loss comes two years later, so its NPV is less negative
at the equity discount rate. The money that is lost is the same, and more of the
strategy's paths lose it. The next sections read where waiting gains and where it costs.

In [ ]:
(
    fixed_outcome.distribution(key="commitment_gap").probability(above="0 AUD"),
    strategy_outcome.distribution(key="commitment_gap").probability(above="0 AUD"),
)

The last cell reads the share of paths with a commitment gap, for the fixed plan
and for the strategy. Section 5 said what the funding check promises, so expect a share
that can be above zero, on either plan.
On the 500 paths it is small, and on those paths each gap comes after the works are paid.
Two effects cause it. A shock lowers the income for some years, and the income does not
cover the loan interest. Or a shock in the last year, with a renewal in that year, makes
the operating cash flow negative: the sale value of lesson 03 capitalises that cash flow,
so the sale value is negative, and the loan goes up at the sale. The funding check does
not close these gaps, because it reads only the budgets on its date.

In [ ]:
helpers.plot_target_curve(
    outcomes={"Strategy": strategy_outcome, "Fixed plan": fixed_outcome},
    key="equity_npv",
)

## 8. Rule parameters and realised decisions

The trigger 1.05 is a parameter of the rule. It is the same on every
path. The start date is a realised decision: it differs from path to path. Read the
realised start years, and the paired difference for each.

In [ ]:
strategy_outcome.distribution(key="start_date").counts()

In [ ]:
difference.by(key="start_date").summary()  # where waiting gains, and where it costs

Read the mean of each row. On the 500 paths, a path that starts in 2029 has a
mean paired difference below zero: the project waits, and then gains less than the fixed
plan. A path that starts later or never has one above zero: on these paths, the market is
weak at the start, and waiting avoids it.

## 9. Optional: several triggers

The trigger is a value in the Specification, so cases can test
several triggers, like any other value: a sensitivity analysis of the rule. Cases
and paths multiply: each trigger runs on every path.

In [ ]:
start_triggers = {
    "1.00": {"start_trigger": 1.00},
    "1.02": {"start_trigger": 1.02},
    "1.05": {"start_trigger": 1.05},
    "1.08": {"start_trigger": 1.08},
}

In [ ]:
trigger_cases = strategy.cases(cases=start_triggers)
trigger_outcome = rk.execution.solve(specification=trigger_cases.across(paths=paths))
trigger_difference = rk.execution.compare(
    outcomes={"Triggers": trigger_outcome, "Fixed plan": fixed_outcome},
    keys=["equity_npv"],
    baseline="Fixed plan",
)
trigger_difference.by_case.summary()

On the 500 paths, the mean paired difference does not move in one direction
with the trigger, and some triggers raise the mean a little. Read the spread as well as the
mean before you choose a trigger.

## Exercises

**Try it 1: a lower trigger.** Start when the index reaches 1.02. How many paths
start in 2027? Does the mean paired difference rise?

In [ ]:
early = strategy.fix(start_trigger=1.02)
early_outcome = rk.execution.solve(specification=early.across(paths=paths))
early_outcome.distribution(key="start_date").counts()

In [ ]:
rk.execution.compare(
    outcomes={"Early": early_outcome, "Fixed plan": fixed_outcome},
    keys=["equity_npv"],
    baseline="Fixed plan",
).summary()

No path starts in 2027: the first check is on 2028-01-01. On the 500 paths, the lower
trigger starts more paths in 2029 and fewer later or never, and the mean paired difference
does not rise: it falls. Section 8 shows why: the paths that start in 2029 are the ones
with a mean paired difference below zero.

**Try it 2: no funding check.** Go back to lesson 04's loan limit, which is where the
commitment gap appears, and remove the funding condition from Building B's rule. On how
many paths does the commitment gap become positive? The commitment gap is the largest
amount by which committed but unpaid costs exceed the available equity and loan
(lesson 03, optional section 11).

In [ ]:
reckless_rule = b_window.where(condition="rent_index >= b_start_trigger").first(
    otherwise=rk.duration.NEVER
)
lesson_04_limit = helpers.load_base_plan()["loan_limit"]
reckless = strategy.fix(b_start=reckless_rule, loan_limit=lesson_04_limit)
reckless_outcome = rk.execution.solve(specification=reckless.across(paths=paths))
gaps = reckless_outcome.distribution(key="commitment_gap")
gaps.probability(above="0 AUD")

In [ ]:
reckless_outcome.distribution(key="b_start").counts()

Each path where Building B starts has a commitment gap. Lesson 04's limit does not
cover the plan's land and works, so without the check the rule commits to what the funding
cannot pay. With the check at that limit, B starts on no path. That is why this lesson
raises the limit.

**Try it 3: a worse site price.** Lower Building B's land share to 25 %. How does
the paired difference change on the paths where B never starts?

In [ ]:
cheap_site = strategy.fix(b_land_share="25 %")
cheap_site_outcome = rk.execution.solve(specification=cheap_site.across(paths=paths))
cheap_difference = rk.execution.compare(
    outcomes={"Cheap site": cheap_site_outcome, "Fixed plan": fixed_outcome},
    keys=["equity_npv"],
    baseline="Fixed plan",
)
cheap_difference.by(key="b_start").summary()  # the row of NEVER: B never starts

On a path that sells the site, a smaller share sells for less. A path that never starts
the project has nothing to sell, and a quick run can have no path in the row of `NEVER`.

## Checks and recap

In [ ]:
assert strategy_outcome.run.paths == fixed_outcome.run.paths == paths.id
for row in strategy_outcome.decisions.observations:
    assert row.available_on <= row.decision_date
decided = rk.execution.DecisionResult.DECIDED
for row in strategy_outcome.decisions.where(key="b_start", result=decided):
    available = row.observed["equity_available"] + row.observed["loan_available"]
    assert available >= row.observed["b_cost"] + row.observed["still_due"]
assert strategy_outcome.case(key=paths.keys[0]).run.rule_evidence.verified

**Recap.**

1. A rule decides a value that the Model relates to the rest of the plan. The Model
   says how dates depend on each other; the Specification says when to commit.
2. A rule sees only what is known at its date. Funding is checked with the decisions
   made so far. The check holds on its date, and a later shock can still take the loan
   above its limit.
3. Compare strategies path by path on the same paths. A rule's parameters are fixed;
   its decisions vary by path. A rule can lower the mean: read the result, and do not
   assume that flexibility pays.

**Next.** Lesson 06 opens the project's structure. Lesson 07 lets rules choose
between designs.